# China A-Share Sector Performance Dashboard

**By: Duoying Gao**  
**Analysis Notebook**  
**Course**: ACC102 - Python Data Product  
**Track**: 4 - Interactive Data Analysis Tool  

------

## 1. Problem Definition & Target Users

**Analysis Question**: How do different industry sectors in China's A-share market perform in terms of returns, volatility, and correlation? Which sectors offer the best risk-adjusted returns, and how do they relate to major market indices?

**Target Users**: 
- Individual investors seeking sector-level insights for portfolio allocation
- Financial analysts comparing sector performance for client recommendations
- Business students learning about Chinese equity market structure
- Fund managers screening sectors for tactical asset allocation

**Why This Matters**: China's A-share market is the second-largest equity market globally. Understanding sector dynamics is essential for making informed investment decisions, managing risk, and identifying opportunities.

## 2. Data Loading & Overview

**Data Source**: Simulated based on AKShare / East Money data structure and real A-share market characteristics  
**Generation Date**: April 2026  
**Coverage**: 10 A-share industry sectors, 6 major market indices, 2023-2026  

The dataset is generated using Python (numpy/pandas) with realistic market microstructure including sector-specific volatility, drift, A-share trading calendar, and simulated market events. The methodology and tool design are fully applicable to real AKShare data.

The dataset includes:
- **Sector Indices**: Banking, Securities, Insurance, Healthcare, New Energy, Defense, Real Estate, Technology, Consumer, Semiconductor
- **Market Indices**: SSE Composite, SZSE Component, ChiNext, SSE 50, CSI 300, CSI 500

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

DATA_DIR = os.path.join('data')

sector_df = pd.read_csv(os.path.join(DATA_DIR, 'sector_indices.csv'))
market_df = pd.read_csv(os.path.join(DATA_DIR, 'market_indices.csv'))

print(f'Sector data shape: {sector_df.shape}')
print(f'Market data shape: {market_df.shape}')
print(f'\nSector columns: {list(sector_df.columns)}')
print(f'\nMarket columns: {list(market_df.columns)}')

In [ ]:
print('Sector Data Overview:')
print(f'  Sectors: {sector_df["sector"].nunique()}')
print(f'  Sector names: {sector_df["sector"].unique().tolist()}')
if '日期' in sector_df.columns:
    print(f'  Date range: {sector_df["日期"].min()} - {sector_df["日期"].max()}')
elif 'date' in sector_df.columns:
    print(f'  Date range: {sector_df["date"].min()} - {sector_df["date"].max()}')
print(f'\nFirst 5 rows of sector data:')
sector_df.head()

In [ ]:
print('Market Index Data Overview:')
print(f'  Indices: {market_df["index_name"].nunique()}')
print(f'  Index names: {market_df["index_name"].unique().tolist()}')
if '日期' in market_df.columns:
    print(f'  Date range: {market_df["日期"].min()} - {market_df["日期"].max()}')
elif 'date' in market_df.columns:
    print(f'  Date range: {market_df["date"].min()} - {market_df["date"].max()}')
print(f'\nFirst 5 rows of market data:')
market_df.head()

## 3. Data Cleaning & Preprocessing

Key cleaning steps:
1. Standardize column names (Chinese to English)
2. Convert date columns to datetime format
3. Handle missing values
4. Calculate derived metrics (daily returns, volatility, moving averages)

In [ ]:
COLUMN_MAP_SECTOR = {
    '日期': 'date', '开盘': 'open', '收盘': 'close',
    '最高': 'high', '最低': 'low', '成交量': 'volume',
    '成交额': 'amount', '振幅': 'amplitude', '涨跌幅': 'pct_change',
    '涨跌额': 'change', '换手率': 'turnover',
}

COLUMN_MAP_MARKET = {
    'date': 'date', 'open': 'open', 'close': 'close',
    'high': 'high', 'low': 'low', 'volume': 'volume',
}

def clean_sector_data(df):
    df = df.copy()
    df.rename(columns={k: v for k, v in COLUMN_MAP_SECTOR.items() if k in df.columns}, inplace=True)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    df.sort_values(['sector', 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

def clean_market_data(df):
    df = df.copy()
    df.rename(columns={k: v for k, v in COLUMN_MAP_MARKET.items() if k in df.columns}, inplace=True)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
    df.sort_values(['index_name', 'date'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

sector_clean = clean_sector_data(sector_df)
market_clean = clean_market_data(market_df)

print(f'Cleaned sector data: {sector_clean.shape}')
print(f'Cleaned market data: {market_clean.shape}')
print(f'\nMissing values in sector data:')
print(sector_clean.isnull().sum()[sector_clean.isnull().sum() > 0])
print(f'\nMissing values in market data:')
print(market_clean.isnull().sum()[market_clean.isnull().sum() > 0])

In [ ]:
# Calculate daily returns for sectors
sector_clean['daily_return'] = sector_clean.groupby('sector')['close'].pct_change() * 100
# Fix first row of each sector (pct_change gives NaN for first row)
sector_clean.loc[sector_clean.groupby('sector').head(1).index, 'daily_return'] = 0

# Calculate 20-day moving average and volatility
sector_clean['ma20'] = sector_clean.groupby('sector')['close'].transform(
    lambda x: x.rolling(window=20, min_periods=10).mean()
)
sector_clean['volatility_20d'] = sector_clean.groupby('sector')['daily_return'].transform(
    lambda x: x.rolling(window=20, min_periods=10).std()
)

# Calculate cumulative returns
sector_clean['cum_return'] = sector_clean.groupby('sector')['daily_return'].transform(
    lambda x: (1 + x / 100).cumprod() - 1
) * 100

print('Derived metrics added: daily_return, ma20, volatility_20d, cum_return')
sector_clean[['sector', 'date', 'close', 'daily_return', 'ma20', 'volatility_20d', 'cum_return']].head(20)

In [ ]:
# Calculate daily returns for market indices
market_clean['daily_return'] = market_clean.groupby('index_name')['close'].pct_change() * 100
market_clean.loc[market_clean.groupby('index_name').head(1).index, 'daily_return'] = 0

market_clean['ma20'] = market_clean.groupby('index_name')['close'].transform(
    lambda x: x.rolling(window=20, min_periods=10).mean()
)

print('Market index derived metrics added')
market_clean.head()

## 4. Descriptive Analysis

In [ ]:
# Sector performance summary
sector_summary = sector_clean.groupby('sector').agg(
    total_days=('date', 'count'),
    avg_daily_return=('daily_return', 'mean'),
    avg_volatility=('volatility_20d', 'mean'),
    total_cum_return=('cum_return', 'last'),
    max_close=('close', 'max'),
    min_close=('close', 'min'),
).round(4)

# Calculate Sharpe-like ratio (return / volatility)
sector_summary['risk_return_ratio'] = (sector_summary['avg_daily_return'] / sector_summary['avg_volatility']).round(4)

sector_summary = sector_summary.sort_values('total_cum_return', ascending=False)
print('Sector Performance Summary:')
sector_summary

In [ ]:
# Market index summary
market_summary = market_clean.groupby('index_name').agg(
    total_days=('date', 'count'),
    avg_daily_return=('daily_return', 'mean'),
    last_close=('close', 'last'),
).round(4)

print('Market Index Summary:')
market_summary

## 5. Visualization & Analysis

In [ ]:
# Sector cumulative returns comparison
fig, ax = plt.subplots(figsize=(14, 7))
for sector in sector_clean['sector'].unique():
    sdata = sector_clean[sector_clean['sector'] == sector]
    ax.plot(sdata['date'], sdata['cum_return'], label=sector, alpha=0.8)

ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (%)')
ax.set_title('Sector Cumulative Returns Comparison')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('figures/sector_cumulative_returns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sector volatility comparison (box plot)
fig, ax = plt.subplots(figsize=(12, 6))
vol_data = sector_clean.dropna(subset=['daily_return'])
sector_order = vol_data.groupby('sector')['daily_return'].std().sort_values(ascending=False).index
sns.boxplot(data=vol_data, x='sector', y='daily_return', order=sector_order, ax=ax)
ax.set_xlabel('Sector')
ax.set_ylabel('Daily Return (%)')
ax.set_title('Sector Daily Return Distribution (Sorted by Volatility)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('figures/sector_volatility_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Risk-Return scatter plot
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    sector_summary['avg_volatility'],
    sector_summary['avg_daily_return'],
    s=200,
    c=sector_summary['total_cum_return'],
    cmap='RdYlGn',
    alpha=0.8,
    edgecolors='black'
)
for idx, row in sector_summary.iterrows():
    ax.annotate(idx, (row['avg_volatility'], row['avg_daily_return']),
                fontsize=9, ha='center', va='bottom')
plt.colorbar(scatter, label='Total Cumulative Return (%)')
ax.set_xlabel('Average 20-Day Volatility (%)')
ax.set_ylabel('Average Daily Return (%)')
ax.set_title('Risk-Return Profile by Sector\nColor = Cumulative Return')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/risk_return_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap between sectors
pivot_returns = sector_clean.pivot_table(index='date', columns='sector', values='daily_return')
corr_matrix = pivot_returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Sector Return Correlation Matrix')
plt.tight_layout()
plt.savefig('figures/sector_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Market index trends
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for idx_name in market_clean['index_name'].unique():
    mdata = market_clean[market_clean['index_name'] == idx_name]
    axes[0].plot(mdata['date'], mdata['close'], label=idx_name, alpha=0.8)

axes[0].set_title('Major Market Index Trends')
axes[0].set_ylabel('Index Level')
axes[0].legend(fontsize=8)

for idx_name in ['CSI 300', 'ChiNext']:
    mdata = market_clean[market_clean['index_name'] == idx_name]
    if not mdata.empty:
        axes[1].plot(mdata['date'], mdata['daily_return'].rolling(20).mean(), label=f'{idx_name} 20D MA', alpha=0.8)

axes[1].set_title('Market Daily Return (20-Day Moving Average)')
axes[1].set_ylabel('Daily Return (%)')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/market_index_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key Findings & Insights

1. **Sector Divergence**: There is significant performance dispersion across sectors. Technology and Semiconductor sectors tend to show higher returns but also higher volatility, while Banking and Insurance offer relatively stable but lower returns.

2. **Risk-Return Trade-off**: The risk-return scatter plot reveals a generally positive relationship between volatility and return, consistent with financial theory. However, some sectors deviate from this pattern, suggesting potential mispricing or sector-specific risks.

3. **Sector Correlations**: Most sectors show positive correlations, particularly within related industries (e.g., Banking and Securities). This has important implications for portfolio diversification — investors should consider adding low-correlation sectors to reduce overall portfolio risk.

4. **Market Benchmark Comparison**: Comparing sector performance against CSI 300 and ChiNext benchmarks helps identify whether a sector is outperforming or underperforming the broader market.

5. **Volatility Clustering**: The box plot analysis shows that Real Estate and Securities sectors exhibit the widest return distributions, indicating higher uncertainty and risk for investors in these sectors.

## 7. Limitations & Future Improvements

- **Limited Time Window**: Only 3 years of data (2023-2026) are included; a longer history would enable more robust trend analysis and cycle identification.
- **Sector-Level Only**: The analysis is at the sector index level; individual stock analysis within sectors could reveal more granular insights.
- **No Fundamental Data**: Only price and volume data are used; incorporating financial ratios (P/E, ROE, etc.) would strengthen the analysis.
- **No Predictive Modeling**: This analysis is purely descriptive; future work could include time-series forecasting or machine learning models.
- **Data Source Limitations**: AKShare data depends on East Money's availability; there may be slight delays or inconsistencies with official exchange data.
- **Future**: Add sector rotation signals, relative strength indicators, and macroeconomic overlay for a more comprehensive investment tool.